# Robustness Check

## Restatement of Main Finding

The primary econometric analysis found little evidence of a strong or statistically significant relationship between changes in international student enrolments and changes in rental prices in New South Wales and Victoria after controlling for population growth, housing supply, COVID effects, and state fixed effects. The coefficient on international student enrolments was economically small and statistically insignificant across specifications. This analysis is interpreted as a descriptive analysis of conditional correlations rather than a causal claim.

## Set of Robustness Checks

To support descriptive claims, I include robustness checks that test whether the conditional correlation between international student enrolments and rental prices is driven by a particular subsample, time window, or functional-form choice. Specifically, I estimate the model separately for NSW and VIC, re-estimate the model without the COVID control, and compare the main differenced-log specification with a non-differenced specification.

### Check 1: HC3 Robust Standard Errors

#### Concern

The main regression may have heteroskedasticity, meaning the variance of the error terms may not be constant across observations. If this happens, the usual OLS standard errors may be unreliable, which can affect the p-values and statistical significance of the coefficients. This is especially important in this project because the sample size is small and some quarters may have larger shocks than others.

#### Approach

To address this concern, the regression is re-estimated using HC3 robust standard errors. HC3 produces more reliable standard error in small samples and under heteroskedasticity. The model specification remains the same as the main regression, but the inference is changed by using HC3 standard errors instead of conventional OLS standard errors.

#### Code 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort properly
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Create COVID dummy
df["COVID"] = ((df["Date"] >= "2020-Q1") & (df["Date"] <= "2022-Q1")).astype(int)

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "COVID",
        "State"
    ]
)

# Robustness Check: HC3 robust standard errors
model_hc3 = smf.ols(
    formula="""
    Q("Δ_ln_rent") ~ Q("Δ_ln_students")
    + Q("Δ_ln_population")
    + Q("Δ_ln_supply")
    + COVID
    + C(State)
    """,
    data=reg_df
).fit(cov_type="HC3")

print(model_hc3.summary())

                            OLS Regression Results                            
Dep. Variable:         Q("Δ_ln_rent")   R-squared:                       0.565
Model:                            OLS   Adj. R-squared:                  0.511
Method:                 Least Squares   F-statistic:                     12.89
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.71e-07
Time:                        21:13:30   Log-Likelihood:                 148.15
No. Observations:                  46   AIC:                            -284.3
Df Residuals:                      40   BIC:                            -273.3
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.0295 

### Check 2: Remove the COVID Dummy

#### Concern


The main regression may rely heavily on the COVID dummy specification. Since the pandemic period caused major disruptions to rental markets, migration flows, border policies, and economic activity, the estimated relationship between international student enrolments and rental prices may be sensitive to how the COVID period is controlled for.

#### Approach 

To test this, the regression is re-estimated after removing the COVID dummy variable while keeping the remaining specification unchanged. If the coefficient on changes in international student enrolments remain economically small and statistically insignificant. This suggests that the main result is not driven solely by the inclusion of the COVID control.

#### Code

In [19]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort properly
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "State"
    ]
)

# Robustness Check: Remove COVID dummy
model_no_covid = smf.ols(
    formula="""
    Q("Δ_ln_rent") ~ Q("Δ_ln_students")
    + Q("Δ_ln_population")
    + Q("Δ_ln_supply")
    + C(State)
    """,
    data=reg_df
).fit()

print(model_no_covid.summary())

                            OLS Regression Results                            
Dep. Variable:         Q("Δ_ln_rent")   R-squared:                       0.504
Model:                            OLS   Adj. R-squared:                  0.456
Method:                 Least Squares   F-statistic:                     10.42
Date:                Mon, 11 May 2026   Prob (F-statistic):           6.43e-06
Time:                        21:14:31   Log-Likelihood:                 145.12
No. Observations:                  46   AIC:                            -280.2
Df Residuals:                      41   BIC:                            -271.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.0195 

### Check 3: Subsample Analysis by State

#### Concern

The main regression pools NSW and VIC into one model. However, the relationship between international student enrolments and rental prices may differ across states because NSW and VIC have different rental markets, population patterns, and student concentrations. Therefore, the pooled result may hide state-specific differences or be driven mainly by one state.

#### Approach

To test this, I re-estimate the main regression separately for NSW and VIC. This keeps the same variables as the main specification, but removes state fixed effects because each regression only contains one state. The purpose is to check whether the estimated relationship differs meaningfully across states.

#### Code

In [20]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort data
df = df.sort_values(["State", "Date"])

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create first-differenced log variables
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Create COVID dummy
df["COVID"] = ((df["Date"] >= "2020-Q1") & (df["Date"] <= "2022-Q1")).astype(int)

# Keep complete observations
reg_df = df.dropna(
    subset=[
        "Δ_ln_rent",
        "Δ_ln_students",
        "Δ_ln_population",
        "Δ_ln_supply",
        "COVID"
    ]
)

# Run separate regressions for NSW and VIC
model_nsw = smf.ols(
    "Q('Δ_ln_rent') ~ Q('Δ_ln_students') + Q('Δ_ln_population') + Q('Δ_ln_supply') + COVID",
    data=reg_df[reg_df["State"] == "NSW"]
).fit()

model_vic = smf.ols(
    "Q('Δ_ln_rent') ~ Q('Δ_ln_students') + Q('Δ_ln_population') + Q('Δ_ln_supply') + COVID",
    data=reg_df[reg_df["State"] == "VIC"]
).fit()

# Print results
print("NSW Only Regression")
print(model_nsw.summary())

print("\n" + "="*80 + "\n")

print("VIC Only Regression")
print(model_vic.summary())

NSW Only Regression
                            OLS Regression Results                            
Dep. Variable:         Q('Δ_ln_rent')   R-squared:                       0.558
Model:                            OLS   Adj. R-squared:                  0.459
Method:                 Least Squares   F-statistic:                     5.675
Date:                Mon, 11 May 2026   Prob (F-statistic):            0.00390
Time:                        21:25:43   Log-Likelihood:                 73.341
No. Observations:                  23   AIC:                            -136.7
Df Residuals:                      18   BIC:                            -131.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept   

### Check 4: Log-Level Specification

#### Concern

The first-differenced log specification may remove important long-run variation and persistent relationships between international student enrolments and rental prices. As a result, the main model may understate broader long-run associations between the variables.

#### Approach

Re-estimate the regression using log-level variables rather than first-differenced logged variables to assess whether the main findings are sensitive to the differencing transformation and the removal of long-run variation.

#### Code

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Create log variables
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# Create COVID dummy
df["COVID"] = (
    (df["Date"] >= "2020-Q1") &
    (df["Date"] <= "2022-Q1")
).astype(int)

# Run logged-level regression
model_logged_levels = smf.ols(
    "ln_rent ~ ln_students + ln_population + ln_supply + C(State) + COVID",
    data=df
).fit()

# Print results
print(model_logged_levels.summary())

                            OLS Regression Results                            
Dep. Variable:                ln_rent   R-squared:                       0.975
Model:                            OLS   Adj. R-squared:                  0.971
Method:                 Least Squares   F-statistic:                     321.3
Date:                Tue, 12 May 2026   Prob (F-statistic):           2.58e-32
Time:                        15:11:36   Log-Likelihood:                 111.85
No. Observations:                  48   AIC:                            -211.7
Df Residuals:                      42   BIC:                            -200.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         -67.3853      4.971    -

### Check 5: Lagged Effect

#### Concern

Rental prices may not respond immediately to changes in international student enrolments within the same quarter. Instead, rental market adjustment may occur with a delay as students arrive, search for accommodation, and enter the housing market. As a result, the contemporaneous specification may understate delayed associations between student enrolments and rental prices.

#### Approach

Re-estimate the main regression using a one-quarter lag of Δ ln(Students) instead of the contemporaneous value. The rest of the specification remains unchanged, including differencing, log transformations, population, housing supply, state fixed effects, and the COVID dummy.

#### Code

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("../data/clean/master_dataset.csv")

# Sort data
df = df.sort_values(["State", "Date"])

# Log transformations
df["ln_rent"] = np.log(df["Rental"])
df["ln_students"] = np.log(df["International Student Enrolments"])
df["ln_population"] = np.log(df["Population"])
df["ln_supply"] = np.log(df["Housing Supply"])

# First differences
df["Δ_ln_rent"] = df.groupby("State")["ln_rent"].diff()
df["Δ_ln_students"] = df.groupby("State")["ln_students"].diff()
df["Δ_ln_population"] = df.groupby("State")["ln_population"].diff()
df["Δ_ln_supply"] = df.groupby("State")["ln_supply"].diff()

# Lagged student variable
df["Δ_ln_students_lag1"] = (
    df.groupby("State")["Δ_ln_students"].shift(1)
)

# COVID dummy
df["COVID"] = (
    (df["Date"] >= "2020-Q1") &
    (df["Date"] <= "2022-Q1")
).astype(int)

# Keep complete observations
reg_df = df.dropna(subset=[
    "Δ_ln_rent",
    "Δ_ln_students_lag1",
    "Δ_ln_population",
    "Δ_ln_supply",
    "COVID"
])

# Regression
model_lag = smf.ols(
    formula="""
    Q("Δ_ln_rent") ~ 
    Q("Δ_ln_students_lag1") +
    Q("Δ_ln_population") +
    Q("Δ_ln_supply") +
    COVID +
    C(State)
    """,
    data=reg_df
).fit()

# Results
print(model_lag.summary())

                            OLS Regression Results                            
Dep. Variable:         Q("Δ_ln_rent")   R-squared:                       0.591
Model:                            OLS   Adj. R-squared:                  0.537
Method:                 Least Squares   F-statistic:                     10.98
Date:                Tue, 12 May 2026   Prob (F-statistic):           1.38e-06
Time:                        14:38:54   Log-Likelihood:                 142.73
No. Observations:                  44   AIC:                            -273.5
Df Residuals:                      38   BIC:                            -262.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Robustness Table 

| Variables                  | (1) Main     | (2) HC3  | (3) No COVID | (4) NSW | (5) VIC | (6) Levels | (7) Lagged |
| -------------------------- | -------------| -------- | -------------| ------- | --------| -----------| -----------|
| Δ ln(Students)             | 0.0003       | 0.0003   | -0.0009      | -0.0003 | 0.0009  | -0.0020    | -0.0009    |
|                            | (0.001)      | (0.001)  | (0.001)      | (0.002) | (0.002) | (0.004)    | (0.001)    |
| Δ ln(Population)           | 1.3973       | 1.3973   | 2.9870       | 1.4137  | 1.6066  | 4.9648     | 1.2495     |
|                            | (1.009)      | (1.009)  | (0.707)      | (2.323) | (1.260) | (0.435)    | (0.886)    |
| Δ ln(Supply)               | -4.1544      | -4.1544  | -3.8533      | -4.3678 | -2.8857 | -0.6652    | -3.4096    |
|                            | (2.595)      | (2.595)  | (2.475)      | (4.338) | (3.863) | (0.286)    | (2.329)    |
| COVID                      | -0.0115      | -0.0115  | —            | -0.0120 | -0.0099 | -0.0067    | -0.0135    |
|                            | (0.005)      | (0.005)  | —            | (0.007) | (0.008) | (0.010)    | (0.004)    |
| State Fixed Effects        | Yes          | Yes      | Yes          | No      | No      | Yes        | Yes        |
| First-Differenced Logs     | Yes          | Yes      | Yes          | Yes     | Yes     | No         | Yes        |
| Robust Standard Errors     | No           | Yes      | No           | No      | No      | No         | No         |
| Sample Restriction         | Full         | Full     | Full         | NSW     | VIC     | Full       | Full       |
| Observations (N)           | 46           | 46       | 46           | 23      | 23      | 48         | 44         |
| R²                         | 0.565        | 0.565    | 0.504        | 0.558   | 0.583   | 0.975      | 0.591      |

**Notes**: The outcome variable is Δ ln(Rent) except in column (6), where the outcome is ln(Rent) in levels. Column (6) uses logged-level variables instead of first-differenced logged variables. All specifications control for population growth and housing supply. Column (2) reports HC3 robust standard errors. Column (3) removes the COVID dummy. Columns (4) and (5) estimate the model separately for NSW and VIC. Column (6) re-estimates the model without first differencing. Column (7) includes a one-quarter lag of Δ ln(Students). State fixed effects are included where indicated. Standard errors are reported in parentheses.

## Interpretation

The robustness checks suggest that the main finding is reasonably stable. Across all specifications, the estimated effect of international student enrolments on rental prices remains economically small and statistically insignificant. Although the sign of the coefficient changes across some specifications, there is no consistent evidence of a strong positive or negative relationship once broader demographic and housing factors are controlled for.

The HC3 robust standard error specification in Column (2) produces more reliable standard error in small samples which is identical to the main model in Column (1), suggesting that heteroskedasticity is not materially affecting inference. The main finding therefore remains credible under alternative standard error assumptions.

Removing the COVID dummy in Column (3) changes the coefficient from slightly positive to slightly negative and lowers the R^2 from 0.565 to 0.504. This suggests that the COVID period captures an important common shock affecting rental markets. However, the student enrolment effect remains economically small, indicating that the baseline result is not entirely dependent on the specific pandemic adjustment.

The NSW-only and VIC-only regressions in Columns (4) and (5) also produce very small coefficients with different signs across states. While this indicates some sensitivity across subsamples, neither state shows a statistically meaningful relationship. This suggests that the main finding is not driven entirely by one state.

The log-level specification in Column (6) produces the largest change, with a much higher R^2 of 0.975 and a larger negative coefficient on student enrolments. This likely reflects strong common upward trends in rents and macroeconomic variables over time rather than a stronger economic relationship. Because the levels model does not remove persistent time trends, the result supports the use of first-differenced logs in the main specification to obtain a more conservative estimate of the short-run relationship.

Finally, the lagged specification in Column (7) still produces a very small and statistically insignificant coefficient, suggesting little evidence that rental prices respond to changes in student enrolments with a one-quarter delay.